# Code to check whether CUDA is available

In [ ]:
import torch
 
print(f"Is CUDA supported by this system? {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
 
# Storing ID of current CUDA device
cuda_id = torch.cuda.current_device()
print(f"ID of current CUDA device: {torch.cuda.current_device()}")
       
print(f"Name of current CUDA device: {torch.cuda.get_device_name(cuda_id)}")

In [ ]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Required libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Code for mixture z-scoring
- Class-wise ratio (weight) values
- Class-wise mean and standard deviation
- Code to apply mixture z-scoring

In [ ]:
def compute_class_weights(labels, n_classes):
    """
    Calculate class weights as the prevalence of each class in the training data.

    Parameters:
    - labels (numpy.ndarray): Array of class labels for each sample.

    Returns:
    - weights (numpy.ndarray): Class weights based on label prevalence.
    """
    n_samples = labels.size
    weights = np.zeros(n_classes)

    for l in range(n_classes):
        weights[l] = np.sum(labels == l) / n_samples

    return weights

def compute_class_statistics(X, labels, n_classes):
    """
    Calculate class-conditional means and variances for each feature.

    Parameters:
    - X (numpy.ndarray): Input data of shape (n_samples, n_features).
    - labels (numpy.ndarray): Array of class labels for each sample.
    - n_classes (int): Number of unique classes.

    Returns:
    - mu (numpy.ndarray): Class-conditional means, shape (n_classes, n_features).
    - sigma2 (numpy.ndarray): Class-conditional variances, shape (n_classes, n_features).
    """
    n_features = X.shape[1]
    mu = np.zeros((n_classes, n_features))
    sigma2 = np.zeros((n_classes, n_features))

    for l in range(n_classes):
        class_data = X[labels == l]
        # Check if there are any samples for this class
        if class_data.size == 0:
            print(f"Warning: No samples found for class {l}.")
            continue  # Skip mean and variance computation for this class
        mu[l] = np.mean(class_data, axis=0)
        sigma2[l] = np.var(class_data, axis=0)

    return mu, sigma2

def mixture_zscore(origin_distribution, train_X, train_labels):
    """
    Perform mixture z-scoring on training and test datasets.

    Parameters:
    - train_X (numpy.ndarray): Training data, shape (n_train_samples, n_features).
    - train_labels (numpy.ndarray): Labels for training data, shape (n_train_samples,).
    - test_X (numpy.ndarray): Test data, shape (n_test_samples, n_features).

    Returns:
    - z_train (numpy.ndarray): Z-scored training data, shape (n_train_samples, n_features).
    - weights (numpy.ndarray): Class weights based on label prevalence.
    """
    # Determine the number of classes and features
    n_classes = len(np.unique(train_labels))
    weights = compute_class_weights(origin_distribution, n_classes)
    # weights = np.array([0.1, 0.35, 0.55])

    # Compute class-conditional statistics on the training set
    mu, sigma2 = compute_class_statistics(train_X, train_labels, n_classes)

    # Calculate mixture z-scores for training set
    weighted_mean_train = np.dot(weights, mu)
    weighted_variance_train = np.dot(weights, sigma2 + (mu - weighted_mean_train) ** 2)
    z_train = (train_X - weighted_mean_train) / np.sqrt(weighted_variance_train)

    # Calculate mixture z-scores for test set using training statistics

    return z_train, weights

def cal_mean_std(folded_data, folded_labels):
     """
     Calculate mean and variance of some labeled epochs for mixture z-scoring.

    Parameters:
    - folded_data (numpy.ndarray): Data to calculate mean and variance, shape (n_train_samples, n_features).
    - folded_labels (numpy.ndarray): Labels to calculate mean and variance, shape (n_train_samples,).

    Returns:
    - mu (numpy.ndarray): Class-conditional means, shape (n_classes, n_features).
    - sigma2 (numpy.ndarray): Class-conditional variances, shape (n_classes, n_features).
    """
     n_classes = len(np.unique(folded_labels))
     mu, sigma2 = compute_class_statistics(folded_data, folded_labels, n_classes)

     return mu, sigma2

def apply_mixture_zscore(test_X, weights, mu, sigma2):
    """
    Perform mixture z-scoring on training and test datasets.

    Parameters:
    - test_X (numpy.ndarray): Testing data, shape (n_train_samples, n_features).
    - test_labels (numpy.ndarray): Labels for testing data, shape (n_train_samples,).
    - weights (numpy.ndarray): Class weights based on training data label prevalence.
    - mu (numpy.ndarray): folded data Class-conditional means, shape (n_classes, n_features).
    - sigma2 (numpy.ndarray): folded data Class-conditional variances, shape (n_classes, n_features).

    Returns:
    - z_test (numpy.ndarray): Z-scored test data, shape (n_test_samples, n_features).
    """

    # Calculate mixture z-scores for training set
    weighted_mean_train = np.dot(weights, mu)
    weighted_variance_train = np.dot(weights, sigma2 + (mu - weighted_mean_train) ** 2)
    z_test = (test_X - weighted_mean_train) / np.sqrt(weighted_variance_train)

    # Calculate mixture z-scores for test set using training statistics

    return z_test

# Train and Test code

In [ ]:
def train_one_epoch(model, optimizer, criterion, dataloader, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device, dtype=torch.float), labels.to(device, dtype=torch.long)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
import sklearn.metrics

def evaluate(model, criterion, dataloader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = np.zeros(3)  # To store correct predictions for each class
    class_total = np.zeros(3)    # To store total predictions for each class
    y_true, y_pred = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device, dtype=torch.float), labels.to(device, dtype=torch.long)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            # Count per-class accuracy
            for i in range(3):  # Assuming 3 classes: REM (0), Wake (1), NREM (2)
                class_correct[i] += ((predicted == i) & (labels == i)).sum().item()
                class_total[i] += (labels == i).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    class_accuracy = class_correct / class_total
    confusion_matrix = sklearn.metrics.confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

    return epoch_loss, epoch_acc, class_accuracy, confusion_matrix

# Code for Within-subject approach
- For 5 days recording per mouse, day 1~3 were used as training data, 4th day was used to calculate mean and standard deviation (for mixture z-scoring), and last day data was use for validation

In [ ]:
from imblearn.over_sampling import SMOTE
# from imblearn.under_sampling import RandomUnderSampler
import Model as Model

def cross_validate_model(all_eeg_data, all_emg_data, mice_labels, device, model_type='1D-CNN', isZscore=False, isMixture=False):
    num_epochs = 5
    batch_size = 128

    # Define data size based on model type
    if model_type == '2D-CNN':
        data_size = all_eeg_data.shape[-2:]  # Shape should be (height, width) for 2D-CNN
        all_emg_data = None  # Ignore EMG data for 2D-CNN as it's already combined in the image
    else:
        data_size = all_eeg_data.shape[-1]

    all_mouse_metrics = []

    # Iterate over each mouse (subject)
    for mouse_idx in range(len(all_eeg_data)):
        print(f"Processing Mouse {mouse_idx + 1}")

        # Select data based on model type
        if model_type == '2D-CNN':
            data = all_eeg_data[mouse_idx]
            shape = (1, *data_size)  # Shape for single image input
        else:
            eeg_data = all_eeg_data[mouse_idx]
            emg_data = all_emg_data[mouse_idx]
            shape = (data_size,)

        mouse_labels = mice_labels[mouse_idx]

        # Split data into training, mean/variance, and validation sets
        if model_type == '2D-CNN':
            train_data = data[:3].reshape(-1, *shape)
            train_labels = mouse_labels[:3].reshape(-1)
            mean_var_data = data[3].reshape(-1, *shape)
            mean_var_labels = mouse_labels[3].reshape(-1)
            val_data = data[4].reshape(-1, *shape)
            val_labels = mouse_labels[4].reshape(-1)
        else:
            eeg_train_data = eeg_data[:3].reshape(-1, *shape)
            emg_train_data = emg_data[:3].reshape(-1, *shape)
            train_labels = mouse_labels[:3].reshape(-1)
            eeg_mean_var_data = eeg_data[3].reshape(-1, *shape)
            emg_mean_var_data = emg_data[3].reshape(-1, *shape)
            mean_var_labels = mouse_labels[3].reshape(-1)
            eeg_val_data = eeg_data[4].reshape(-1, *shape)
            emg_val_data = emg_data[4].reshape(-1, *shape)
            val_labels = mouse_labels[4].reshape(-1)
        original_train_labels = train_labels


        # Apply SMOTE to balance classes in training data
        if model_type == '2D-CNN':
            smote = SMOTE(sampling_strategy='auto', random_state=42)
            train_data, train_labels = smote.fit_resample(train_data.reshape(-1, train_data.shape[-2]*train_data.shape[-1]), train_labels)
            train_data = train_data.reshape(-1, *shape)
        else:
            eeg_smote = SMOTE(sampling_strategy='auto', random_state=42)
            eeg_train_data, eeg_train_labels = eeg_smote.fit_resample(eeg_train_data, train_labels)
            emg_smote = SMOTE(sampling_strategy='auto', random_state=42)
            emg_train_data, emg_train_labels = emg_smote.fit_resample(emg_train_data, train_labels)
            if np.array_equal(eeg_train_labels, emg_train_labels):
                train_labels = eeg_train_labels
    
        # Print class distribution for training and validation data
        unique_train, counts_train = np.unique(train_labels, return_counts=True)
        unique_val, counts_val = np.unique(val_labels, return_counts=True)
        print(f"  Class distribution for Training Data: {dict(zip(unique_train, counts_train))}, Validation Data: {dict(zip(unique_val, counts_val))}")

        # Apply Z-score normalization if specified
        if isZscore:
            scaler = StandardScaler()
            if model_type == '2D-CNN':
                train_data = scaler.fit_transform(train_data.reshape(-1, train_data.shape[-2]*train_data.shape[-1])).reshape(train_data.shape)
            else:
                eeg_train_data = scaler.fit_transform(eeg_train_data)
                emg_train_data = scaler.fit_transform(emg_train_data)

        # Apply mixture z-score if specified
        if isMixture:
            if model_type == '2D-CNN':
                train_data, train_weights = mixture_zscore(original_train_labels, 
                                                           train_data.reshape(-1, train_data.shape[-2] * train_data.shape[-1]), train_labels).reshape(train_data.shape)
            else:
                eeg_train_data, eeg_weights = mixture_zscore(original_train_labels, eeg_train_data, train_labels)
                emg_train_data, emg_weights = mixture_zscore(original_train_labels, emg_train_data, train_labels)


        # Calculate mean and variance using day 4 data for normalization
        if model_type == '2D-CNN':
            # Reshape to (batch, height * width) for normalization
            reshaped_data = mean_var_data.reshape(-1, mean_var_data.shape[-2] * mean_var_data.shape[-1])
            mu, sigma2 = cal_mean_std(reshaped_data, mean_var_labels)
        else:
            eeg_mu, eeg_sigma2 = cal_mean_std(eeg_mean_var_data, mean_var_labels)
            emg_mu, emg_sigma2 = cal_mean_std(emg_mean_var_data, mean_var_labels)

        classes_weights = compute_class_weights(original_train_labels, 3)
        
        # Initialize model, loss function, and optimizer
        if model_type == '1D-CNN':
            model = Model.CNN1DModel().to(device)
            train_dataset = Model.CNN1D(eeg_train_data.reshape(-1, 1, data_size), emg_train_data.reshape(-1, 1, data_size), train_labels)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.0001)
        elif model_type == '2D-CNN':
            model = Model.SSANN().to(device)
            train_dataset = Model.SSDataset(train_data, train_labels)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.SGD(model.parameters(), lr=0.015, momentum=0.9)
        elif model_type == 'CNN+BiLSTM':
            model = Model.DeepSleepNet(n_outputs=3, return_feats=False, n_chans=2, chs_info=None, n_times=1280, input_window_seconds=2.5, sfreq=512, n_classes=None).to(device)
            train_dataset = Model.CNNBiLSTM(eeg_train_data.reshape(-1, 1, data_size), emg_train_data.reshape(-1, 1, data_size), train_labels)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.05)
        else:
            raise ValueError("Model type not recognized!")
    
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # Train the model
        initial_learning_rate = 0.015
        for epoch in range(num_epochs):
            if model_type == '2D-CNN':
                current_learning_rate = initial_learning_rate * (0.85 ** epoch)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_learning_rate
            train_loss, train_acc = train_one_epoch(model, optimizer, criterion, train_loader, device)
            print(f"Epoch {epoch + 1}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}")

        # Prepare validation data
        if isZscore:
            if model_type == '2D-CNN':
                val_data = scaler.transform(val_data.reshape(-1, val_data.shape[-2]*val_data.shape[-1])).reshape(val_data.shape)
            else:
                eeg_val_data = scaler.transform(eeg_val_data)
                emg_val_data = scaler.transform(emg_val_data)

        if isMixture:
            if model_type == '2D-CNN':
                val_data = apply_mixture_zscore(val_data.reshape(-1, val_data.shape[-2] * val_data.shape[-1]), train_weights, mu, sigma2).reshape(val_data.shape)
            else:
                eeg_val_data = apply_mixture_zscore(eeg_val_data, eeg_weights, eeg_mu, eeg_sigma2)
                emg_val_data = apply_mixture_zscore(emg_val_data, emg_weights, emg_mu, emg_sigma2)

        # Create Dataset and DataLoader for validation
        if model_type == '1D-CNN':
            val_dataset = Model.CNN1D(eeg_val_data.reshape(-1, 1, data_size), emg_val_data.reshape(-1, 1, data_size), val_labels)
        elif model_type == '2D-CNN':
            val_dataset = Model.SSDataset(val_data, val_labels)
        elif model_type == 'CNN+BiLSTM':
            val_dataset = Model.CNNBiLSTM(eeg_val_data.reshape(-1, 1, data_size), emg_val_data.reshape(-1, 1, data_size), val_labels)
        else:
            raise ValueError("Model type not recognized!")
        
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Evaluate the model
        val_loss, val_acc, class_acc, confusion_mat = evaluate(model, criterion, val_loader, device)
        print(f"    Validation Loss = {val_loss:.4f}, Validation Acc = {val_acc:.4f}")
        print(f"    Class Accuracies: REM = {class_acc[0]:.4f}, Wake = {class_acc[1]:.4f}, NREM = {class_acc[2]:.4f}")

        # Store metrics for this mouse
        all_mouse_metrics.append((val_loss, val_acc, class_acc, confusion_mat))

    # Average metrics across all mice
    avg_val_loss = np.mean([metrics[0] for metrics in all_mouse_metrics])
    avg_val_acc = np.mean([metrics[1] for metrics in all_mouse_metrics])
    avg_class_acc = np.mean([metrics[2] for metrics in all_mouse_metrics], axis=0)

    print(f"Average Validation Loss: {avg_val_loss:.4f}")
    print(f"Average Validation Accuracy: {avg_val_acc:.4f}")
    print(f"Average Class Accuracies: REM = {avg_class_acc[0]:.4f}, Wake = {avg_class_acc[1]:.4f}, NREM = {avg_class_acc[2]:.4f}")

    # Return metrics for all mice
    return all_mouse_metrics

# Run code and save result metrics
- model = '1D-CNN' ('2D-CNN', 'CNN+BiLSTM')
- Change the parameter to choose which model to be run

In [ ]:
model_type = 'AttnSleep'
base_path = 'Result_metrics/within'
# Set up the save path based on the model type
if model_type == '1D-CNN':
    model_folder = os.path.join(base_path, '1D-CNN')
elif model_type == '2D-CNN':
    model_folder = os.path.join(base_path, '2D-CNN')
elif model_type == 'CNN+BiLSTM':
    model_folder = os.path.join(base_path, 'CNN+BiLSTM')
else:
    raise ValueError("Model type not recognized!")

# Create the directory if it doesn’t exist
os.makedirs(model_folder, exist_ok=True)

# Load raw data
- open-source dataset (10 mice EEG and EMG)
- 512 Hz sampling dataset
- segmented into 2.5 seconds (labeled in 2.5s segments)

In [ ]:
if model_type == '2D-CNN':
    all_data = np.load("path/to/your/epoched_spectrogram_data.npy") # Spectrogram data already preprocessed for 2D-CNN
    print(all_data.shape)
else:
    all_eeg_data = np.load("path/to/your/epoched_eeg_data.npy")
    all_emg_data = np.load("path/to/your/epoched_emg_data.npy")
    print(all_eeg_data.shape)
    print(all_emg_data.shape)
all_labels = np.load("path/to/your/label_data.npy").squeeze() - 1
print(all_labels.shape)

In [ ]:
# Perform cross-validation
if model_type == '2D-CNN':
    metrics_nonorm = cross_validate_model(all_data, None, all_labels, device, model_type=model_type, isZscore=False, isMixture=False)
else:
    metrics_nonorm = cross_validate_model(all_eeg_data, all_emg_data, all_labels, device, model_type=model_type, isZscore=False, isMixture=False)

# Save the metrics using pickle
with open(os.path.join(model_folder, 'matrix_no_norm.pkl'), 'wb') as f:
    pickle.dump(metrics_nonorm, f)

print(f"Metrics saved for {model_type} in {model_folder}")

- 학습에 신간 소요가 길어 한번에 한나만 학습, 나머지는 comment

In [ ]:
# if model_type == '2D-CNN':
#     metrics_zscore = cross_validate_model(all_data, None, all_labels, device, model=model_type, isZscore=True, isMixture=False)
# else:
#     metrics_zscore = cross_validate_model(all_eeg_data, all_emg_data, all_labels, device, model=model_type, isZscore=True, isMixture=False)

# with open(os.path.join(model_folder, 'matrix_z_scoring.pkl'), 'wb') as f:
#     pickle.dump(metrics_zscore, f)

# print(f"Metrics saved for {model_type} in {model_folder}")

In [ ]:
# if model_type == '2D-CNN':
#     metrics_mixture = cross_validate_model(all_data, None, all_labels, device, model=model_type, isZscore=False, isMixture=True)
# else:
#     metrics_mixture = cross_validate_model(all_eeg_data, all_emg_data, all_labels, device, model=model_type, isZscore=False, isMixture=True)

# with open(os.path.join(model_folder, 'matrix_mixture_z_scoring.pkl'), 'wb') as f:
#     pickle.dump(metrics_mixture, f)

# print(f"Metrics saved for {model_type} in {model_folder}")